In [1]:
import os 
from dotenv import load_dotenv
from dbrepo.RestClient import RestClient
from dbrepo.api.dto import CreateTable, CreateTableColumn, CreateTableConstraints, CreateForeignKey

load_dotenv()
password = os.getenv("DBREPO_PASS")
username = os.getenv("DBREPO_USER")
client = RestClient("https://test.dbrepo.tuwien.ac.at/", username=username, password=password)

def create_city_map_table(database_id):
    """This function collects resources required for creating the manually-compiled mapping table for cities."""
    cols = [
        CreateTableColumn(name="nuts_code", type="varchar", size=5, primary_key=True, null_allowed=False,
                        #concept_uri="http://purl.org/linked-data/sdmx/2009/dimension#refArea",
                        description="5-character NUTS-3 administrative code (e.g., AT221): https://ec.europa.eu/eurostat/web/nuts"),
                        
        CreateTableColumn(name="city_name", type="varchar", size = 100, primary_key=False, null_allowed=False,
                        #concept_uri= "http://purl.obolibrary.org/obo/NCIT_C95378",
                        description="The name of the city from EUDA/SCODA data (e.g., Graz)"),
    ]

    # define constraints
    cons_city = CreateTableConstraints(primary_key=["nuts_code"], uniques=[["city_name"]])

    df_city = CreateTable(
        name="city_map",
        description="This table serves as the bridge/mapping schema. It resolves the city names used by the EUDA to the NUTS-3 codes used by Eurostat.",
        columns=cols,
        constraints=cons_city,
        is_public=True,
        is_schema_public=True
    )

    response = client._wrapper(
        method="post", 
        url=f'/api/v1/database/{database_id}/table', 
        payload=df_city
    )

    print(f"Response Status: {response.status_code}")
    if response.status_code == 201:
        print("Success! Table created.")
    else:
        print(response.text)

def create_gdp_table(database_id):
    """This function collects resources required for creating the GDP table."""
    cols_gdp = [
        CreateTableColumn(name="nuts_code", type="varchar", size=5, primary_key=True, null_allowed=False,
                        description="5-character NUTS-3 administrative code (e.g. AT221): https://ec.europa.eu/eurostat/web/nuts"),
        CreateTableColumn(name="ref_year", type="int", primary_key=True, null_allowed=False,
                        #concept_uri= "http://rs.tdwg.org/dwc/terms/year",
                        description="4-digit year of the record: 2011 to 2024"),
        CreateTableColumn(name="gdp", type="decimal", size = 30, d=2, primary_key=False, null_allowed=True,
                        #concept_uri= "http://purl.org/linked-data/sdmx/2009/measure#obsValue",
                        #unit_uri= "https://www.omg.org/spec/Commons/QuantitiesAndUnits/QuantityValue",
                        description="Gross Domestic Product by NUTS Code"),
        CreateTableColumn(name="currency", type="varchar", size = 10, primary_key=False, null_allowed=True,
                        #concept_uri= "http://purl.org/linked-data/sdmx/2009/attribute#currency",
                        #unit_uri= "https://www.omg.org/spec/Commons/QuantitiesAndUnits/hasUnit",
                        description="Currency pertaining to the Gross Domestic Product (in gdp column)")
    ]

    foreign_keys_gdp = [
        CreateForeignKey(
            columns=["nuts_code"],           
            referenced_table="city_map", 
            referenced_columns=["nuts_code"] 
        )
    ]

    cons_gdp = CreateTableConstraints(
        primary_key=["nuts_code", "ref_year"],
        foreign_keys=foreign_keys_gdp
    )

    df_gdp = CreateTable(
        name="gdp_data",
        description="This table stores the economic baseline for European regions (gross domestic product at current market prices by NUTS3 regions). Sourced from Eurostat.",
        columns=cols_gdp,
        constraints=cons_gdp,
        is_public=True,
        is_schema_public=True
    )

    response = client._wrapper(
        method="post", 
        url=f'/api/v1/database/{database_id}/table', 
        payload=df_gdp
    )
    print(response)

def create_wastewater_table(database_id):
    """This function collects resources required for creating the wastewater metabolite concentration table."""
    cols = [
        CreateTableColumn(name="city_name", type="varchar", size = 100, primary_key=True, null_allowed=False,
                        description="The name of the city from EUDA/SCODA data (e.g., Graz)"),
        CreateTableColumn(name="ref_year", type="int", primary_key=True, null_allowed=False,
                        description="4-digit year of the record: 2011 to 2024"),
        CreateTableColumn(name="metabolite_name", type="varchar", size=100, primary_key=True, null_allowed=False,
                        #concept_uri= "http://purl.obolibrary.org/obo/CHEBI_23367",        
                        description="The specific substance whose concentration was estimated (e.g., Cocaine, MDMA)"),
        CreateTableColumn(name="daily_mean_concentration", type="decimal", size=15, d=2, primary_key=False, null_allowed=True,
                        #concept_uri= "http://purl.allotrope.org/ontologies/process#AFP_0002800",
                        #unit_uri= "https://www.omg.org/spec/Commons/QuantitiesAndUnits/DerivedUnit",
                        description="(mg/1000person/day) Daily averages of metabolite concentration scaled by the population estimates. Values below the method limit of quantification are indicated as zero.")
    ]

    foreign_keys_waste = [
        CreateForeignKey(
            columns=["city_name"],           
            referenced_table="city_map", 
            referenced_columns=["city_name"] # Points to the Unique column
        )
    ]

    cons_waste = CreateTableConstraints(
        primary_key=["city_name", "ref_year", "metabolite_name"],
        foreign_keys=foreign_keys_waste
    )

    df_wastewater_data = CreateTable(
        name="wastewater_data",
        description="Estimated concentrations of metabolites in municipal wastewater for various cities over the period of 2011-2024. Sourced from EUDA and SCORE",
        columns=cols,
        constraints=cons_waste,
        is_public=True,
        is_schema_public=True
    )

    # call wrapper with the object
    response = client._wrapper(
        method="post", 
        url=f'/api/v1/database/{database_id}/table', 
        payload=df_wastewater_data
    )

    print(f"Response Status: {response.status_code}")
    if response.status_code == 201:
        print("Success! Table created.")
    else:
        print(response.text)

In [2]:
DB_ID = os.getenv("DB_ID") 

create_city_map_table(database_id = DB_ID)
create_gdp_table(database_id = DB_ID)
create_wastewater_table(database_id = DB_ID)

Response Status: 201
Success! Table created.
<Response [201]>
Response Status: 201
Success! Table created.


In [3]:
from dbrepo.api.dto import (
    CreateIdentifier,
    CreateIdentifierTitle,
    CreateIdentifierDescription,
    RelatedIdentifier,
    RelatedIdentifierType,
    RelatedIdentifierRelation,
    CreateIdentifierCreator,
    CreateRelatedIdentifier,
    DescriptionType,
    IdentifierType,
    License,
    Language
)

def build_consolidated_use_case_pid(database_id) -> CreateIdentifier:
    """
    Generates DBRepo metadata
    """
    
    titles = [
        CreateIdentifierTitle(
            title="Predictive Modeling of Regional GDP per Capita based on Wastewater-Based Epidemiology",
            language=Language.EN
        )
    ]
    
    # 2. Comprehensive Metadata Descriptions (Abstract, Methods, Scope, Units)
    descriptions = [
        CreateIdentifierDescription(
            description=(
                """Abstract: This use case explores the correlation relationship between 
                illicit drug use in major European cities and their regional economic productivity (GDP per capita). 
                Original Publishers: EUDA & SCORE, EUROSTAT.
                EUDA & SCORE: URI: https://www.euda.europa.eu/data/repository/drugs-municipal-wastewater-europe-source-data-2026_en
                EUROSTAT: DOI: https://doi.org/10.2908/NAMA_10R_3GDP, URI: https://ec.europa.eu/eurostat/databrowser/view/nama_10r_3gdp__custom_20659344/default/table"""
            ),
            type=DescriptionType("Abstract"),
            language=Language.EN
        ),
        CreateIdentifierDescription(
            description=(
                """Data Stewardship and Preprocessing Challenge: While the drug dataset identifies locations 
                by specific city strings (e.g., 'Graz', 'Steyr'), Eurostat utilizes standardized NUTS-3 administrative codes 
                (e.g., 'DE212'). This is resolved via a custom mapping schema table ('city_map'.
                Only the active filtered subset utilized in this longitudinal frame is republished here."""
            ),
            type="Methods",
            language=Language.EN
        ),
        CreateIdentifierDescription(
            description=(
                """Temporal & Spatial Coverage: 
                Wastewater tracking spans annually from 2011 to 2025 across 115 cities and 25 countries in the European Union, 
                Norway, and Türkiye. GDP tracking spans annually from 2000 to 2024 across EU Member States, Candidate and 
                potential Candidate Countries, Norway, and Switzerland."""
            ),
            type=DescriptionType("TechnicalInfo"),
            language=Language.EN
        ),
        CreateIdentifierDescription(
            description=(
                """Units of Measure: "
                Wastewater metrics indicate concentrations (mg/1000p/day) of illicit drug loads (Cocaine, Methamphetamine, MDMA) 
                measured from 24-hour composite samples collected over a single week between March and May. 
                GDP values indicate economic output expressed in National Currency, Euros, Purchasing Power Standards (PPS), 
                thousands of persons/hours worked, growth rates, or Index 2020=100."""
            ),
            type=DescriptionType("Other"),
            language=Language.EN
        )
    ]
    
    # 3. Explicit Provenance & Lineage Relationships
    related_identifiers = [
        # Upstream Eurostat Source Dataset
        CreateRelatedIdentifier(
            id="10.2908/NAMA_10R_3GDP",
            value="10.2908/NAMA_10R_3GDP",
            type=RelatedIdentifierType.DOI,
            relation=RelatedIdentifierRelation.IS_DERIVED_FROM
        ),
        # Upstream EUDA Open Repository Source
        CreateRelatedIdentifier(
            id="https://www.euda.europa.eu/data/repository/drugs-municipal-wastewater-europe-source-data-2026_en",
            value="https://www.euda.europa.eu/data/repository/drugs-municipal-wastewater-europe-source-data-2026_en",
            type=RelatedIdentifierType.URL,
            relation=RelatedIdentifierRelation.IS_DERIVED_FROM
        )
    ]

    cc_by_4_0 = License(
        identifier="CC-BY-4.0",
        uri="https://creativecommons.org/licenses/by/4.0/",
        description=(
            "Creative Commons Attribution 4.0 International: Allows users to copy, "
            "distribute, display, perform, and modify the work, even for commercial purposes, "
            "provided that they give appropriate credit to the original creator."
        )
    )
    
    # 4. Master DataCite Payload Assembly
    identifier_payload = CreateIdentifier(
        database_id=database_id,
        publication_year=2026,           # Project release date
        publisher="EUDA & SCORE, EUROSTAT",
        type = IdentifierType.DATABASE,
        language=Language.EN,
        licenses=[cc_by_4_0],          # Explicitly declared open license for both sources
        titles=titles,
        descriptions=descriptions,
        related_identifiers=related_identifiers,
        
        # Comprehensive project curator/author roster mapped to standard DataCite creator formats
        creators=[
            CreateIdentifierCreator(creator_name="Helene Vaught", firstname= "Helene", lastname= "Vaught", affiliation= "TU Wien"),
            CreateIdentifierCreator(creator_name="Vlada Hlushchenko", firstname= "Vlada", lastname= "Hlushchenko", affiliation= "TU Wien"),
            CreateIdentifierCreator(creator_name="Barnabás Paksi", firstname= "Barnabás", lastname= "Paksi", affiliation= "TU Wien"),
            CreateIdentifierCreator(creator_name="Amélie Assmayr", firstname= "Amélie", lastname= "Assmayr", affiliation= "TU Wien")
        ]
    )
    
    return identifier_payload

In [ ]:
identifier_payload = build_consolidated_use_case_pid(database_id=DB_ID)